<a href="https://colab.research.google.com/github/redinbluesky/handson-llm/blob/main/06_프롬프트_엔지니어링.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#  목차
* [Chapter 6 서론](#chapter6)
* [Chapter 6-1 텍스트 생성 모델 사용하기](#chapter6-1)
    * [Chapter 6-1-1 텍스트 생성 모델 선택하기](#chapter6-1-1)
    * [Chapter 6-1-2 텍스트 생성 모델 로드하기](#chapter6-1-2)
    * [Chapter 6-1-3 모델 출력 제어하기](#chapter6-1-3)
 * [Chapter 6-2 프롬프트 엔지니어링 소개](#chapter6-2)   
     * [Chapter 6-2-1 프롬프트의 기본 구성요소](#chapter6-2-1)   

## Chapter 6 서론 <a class="anchor" id="chapter6"></a>
1. 사전 훈련된 트랜스포머 기반 모델은 사용자의 프롬프트에 대한 응답으로 텍스트를 생성하는 능력이 뛰어나다.

2. 프롬프트 엔지니어링은 생성된 텍스트의 품질을 향상시키기 위해 프롬프트를 설계하는 방법이다.

## Chapter 6-1 텍스트 생성 모델 사용하기 <a class="anchor" id="chapter6-1"></a>
### Chapter 6-1-1 텍스트 생성 모델 선택하기 <a class="anchor" id="chapter6-1-1"></a>
1. 텍스트 모델을 선택하기 전에 독점 모델을 사용할지 오픈 소스 모델을 사용할 지 결정해야한다.
    - 독점 모델은 일반적으로 더 강력하지만 비용이 많이 들고, 오픈 소스 모델은 더 유연하고 자유롭게 사용할 수 있다.

2. 일반적으로 유명한 파운데이션 모델은 아래의 이미지와 같다.

    ![파운데이션 모델](./image/06_foundation_models.png)

3. 이런 파운데이션 모델을 사용해 미세 튜닝된 모델은 수천개 이상 존재하며, 각각 특정 작업에 최적화 되어있다.
    - 이런 모델 중 어떤 모델을 선택해야하는 지는 사용자의 작업과 요구사항에 따라 다르다.

4. 작은 크기의 파운데이션 모델에서 시작하는 것이 좋다.
    - Phi-3-mini는 작은 VRM이 설치된 장치에서 사용할 수 있다.

5. 일반적으로 작은 모델에서 스케일을 확대하는 것이 축소하는 것보다 용이하다.
    - 작은 모델에서 원하는 결과를 얻을 수 있다면, 더 큰 모델에서도 원하는 결과를 얻을 가능성이 높다.

### Chapter 6-1-2 텍스트 생성 모델 로드하기 <a class="anchor" id="chapter6-1-2"></a>

In [2]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

# Phi-3-mini 모델과 토크나이저 로드
model = AutoModelForCausalLM.from_pretrained("microsoft/Phi-3-mini-4k-instruct"
                                            , device_map="cuda",
                                            dtype="auto")

tokenizer = AutoTokenizer.from_pretrained("microsoft/Phi-3-mini-4k-instruct")

# 텍스트 생성 파이프라인 생성
pip = pipeline("text-generation", model=model, tokenizer=tokenizer, return_full_text=False, max_new_tokens=500, do_sample=False)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Device set to use cuda
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


In [3]:
# 프롬프트
message = [{"role": "user", "content": "Create a funny joke about chickens"},]

# 텍스트 생성
output = pip(message)
print(output[0]['generated_text'])

 Why did the chicken join the band? Because it had the drumsticks!


In [7]:
# 프롬프트 템플릿을 적용합니다.
prompt = pip.tokenizer.apply_chat_template(message, tokenize=False)
print(prompt)

<|user|>
Create a funny joke about chickens<|end|>
<|endoftext|>


1. 프롬프트 템플릿은 모델을 훈련하는데 사용된다.
    - 누가 무엇을 말했는지에 대한 정보와 모델이 어떻게 응답해야하는지에 대한 정보를 포함한다.

2. 프롬프트 템플릿을 이미지로 표현하면 아래와 같다.

    ![프롬프트 템플릿](./image/06_prompt_template.png)

### Chapter 6-1-3 모델 출력 제어하기 <a class="anchor" id="chapter6-1-3"></a>
1. 모델의 매개변수를 조정하여 출력을 제아할 수 있다.
    - 예를 들어, `temperature` 매개변수는 모델의 출력을 더 창의적이거나 보수적으로 만들 수 있다.
    - `top_k`와 `top_p` 매개변수는 모델이 다음 단어를 선택할 때 고려하는 후보 단어의 수를 제한한다.

2. "I am driving a"란 문장된에 "car"나 "trcuk"과 같은 단어가 나올 확율이 "elephant"보다 높지만, 매우 낮더라도 "elephant"가 생성될 가능성은 있다.

3. do_sample=False로 설정하면 모델이 항상 가장 높은 확률을 가진 단어를 선택하도록 강제할 수 있다.

4. temperature 매개변수는 텍스트 생성의 무작위성 또는 창의성을 조절한다.
    - 확률이 낮은 토큰을 선택할 가능성이 얼마인지 결정한다.
    - 0에 가까운 값은 모델이 가장 확률이 높은 토큰을 선택하도록 강제한다.


In [ ]:
# 높은 temperature로 텍스트 생성
# 실행할 때마다 출력이 바뀐다.
output = pip(message, temperature=1.0, do_sample=True)
print(output[0]['generated_text'])

 Why did the chicken join the symphony orchestra? Because she had good chicken beats!


5. top_p는 LLM이 고려할 토큰 일부를 제어하는 샘플링 기법이다.
    - top_p에 지정한 누적 확률에 도달할 때까지 후보 토큰을 모은다.
    - 0.1로 설정하면 모델이 다음 단어를 선택할 때 가장 확률이 높은 10%의 토큰만 고려한다.
    - 아래의 이미지에서 보듯이 top_p가 낮을수록 모델이 다음 단어를 선택할 때 고려하는 후보 토큰의 수가 줄어든다.
    
        ![top_p](./image/06_top_p.png)

6. top_k는 모델이 고려할 수 있는 토큰의 수를 제어한다.
    - top_k=50으로 설정하면 모델이 다음 단어를 선택할 때 가장 확률이 높은 50개의 토큰만 고려한다.
    - top_k가 낮을수록 모델이 다음 단어를 선택할 때 고려하는 후보 토큰의 수가 줄어든다.

In [9]:
# 높은 top_p로 텍스트 생성
output = pip(message, top_p=1, do_sample=True)
print(output[0]['generated_text'])

 Why did the chicken join a book club? Because it wanted to improve its egg-scape!


7. temperature, top_p 값 사용 예시는 아래의 이미지와 같다.

    ![temperature_top_p](./image/06_temperature_top_p.png)

## Chapter 6-2 프롬프트 엔지니어링 소개 <a class="anchor" id="chapter6-2"></a>
1. 프롬프트 엔지니어링의 주요 목적은 모델로부터 유용한 응답을 얻는 것이다.

2. 프롬프트 최적화는 반복적인 과정이 필요하다.

## Chapter 6-2-1 프롬프트의 기본 구성요소<a class="anchor" id="chapter6-2-1"></a>
1. LLM은 아래의 그림과 같이 프롬프트를 통해 특정 작업을 요청하지 않으면 기본적으로 이전 단어를 기반으로 이후에 나올 단어를 예측한다.

    ![프롬프트 예측](./image/06_prompt_prediction.png)

 2. LLM에 한 문장이 긍정적인지 부정적인지를 분류한다고 가정해보자
    - 프롬프트는 지시와 이에 관련된 두 가지 데이터로 구성될 수 있다.
    
      ![프롬프트 구성요소](./image/06_prompt_components.png)

3. 모델이 '긍정적', '부정적' 만을 출력하게 하려면 프롬프트에 명확한 지시어가 필요하다.
    - 모델은 이런 요소에 직접 훈련되지 않았지만 이런 구조를 일반화할 수 있는 충분한 지시문을 학습했다.
    
        ![프롬프트 지시어](./image/06_prompt_instruction.png)


4. 원하는 응답을 얻을 때까지 프롬프트에 요소를 추가하거나 업데이트할 수 있다.
    - 예시를 추가, 사용 사례 설명, 추가적인 맥락제공 등.
    - 이런 구성 요소를 설계하는 창의설이 핵심이다.